# EA2 - Actividad 2.2: Transformacion y Limpieza de Datos

## Objetivos
- Detectar y manejar valores nulos (nulls) en DataFrames
- Identificar y eliminar registros duplicados
- Realizar casting y conversion de tipos de datos
- Crear funciones definidas por el usuario (UDFs)
- Combinar multiples fuentes de datos con join y union

## Conceptos Clave

### Calidad de Datos en un Pipeline ETL

En un pipeline ETL (Extract, Transform, Load), la fase de **transformacion** es donde se limpia y prepara los datos. Los problemas mas comunes son:

| Problema | Descripcion | Solucion |
|----------|-------------|----------|
| **Valores nulos** | Campos vacios o sin datos | `dropna()`, `fillna()` |
| **Duplicados** | Registros repetidos | `dropDuplicates()` |
| **Tipos incorrectos** | Columna numerica leida como string | `cast()`, `to_date()` |
| **Formatos inconsistentes** | Mezcla de mayusculas/minusculas | `lower()`, `upper()`, `trim()` |
| **Valores fuera de rango** | Datos que no tienen sentido | `filter()`, `when()` |

### UDFs (User Defined Functions)

Cuando la logica de transformacion es compleja y no se puede expresar con funciones built-in de Spark, se pueden crear **UDFs**. Sin embargo, las UDFs son mas lentas que las funciones nativas porque Spark no puede optimizarlas.

**Regla:** Siempre preferir funciones nativas de Spark (`F.when()`, `F.col()`, etc.) sobre UDFs cuando sea posible.

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("EA2_02_transformacion_limpieza") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Cargar los Datos

Trabajaremos con dos datasets: `sales.csv` (ventas semanales por tienda y departamento) y `stores.csv` (informacion de las tiendas).

In [ ]:
# Leer los datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

print("=== Sales ===")
df_sales.printSchema()
df_sales.show(5)
print(f"Filas en sales: {df_sales.count()}")

print("\n=== Stores ===")
df_stores.printSchema()
df_stores.show(5)
print(f"Filas en stores: {df_stores.count()}")

## 2. Detectar Valores Nulos

Antes de limpiar, necesitamos saber **donde** estan los problemas. Esta consulta cuenta los nulls por columna.

In [ ]:
# Contar valores nulos por columna en sales
print("Valores nulos por columna en sales:")
df_sales.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_sales.columns]
).show()

# Tambien podemos ver el porcentaje de nulls
total_filas = df_sales.count()
print("Porcentaje de nulls por columna:")
df_sales.select(
    [F.round(F.count(F.when(F.isnull(c), c)) / total_filas * 100, 2).alias(c) for c in df_sales.columns]
).show()

## 3. Manejar Valores Nulos

Hay dos estrategias principales:
- **Eliminar filas** con `dropna()`: cuando los nulls son pocos y no queremos inventar datos
- **Rellenar valores** con `fillna()`: cuando queremos mantener las filas y usar un valor por defecto

In [ ]:
# Estrategia 1: Eliminar filas con nulls en columnas criticas
df_sin_nulls = df_sales.dropna(subset=["Weekly_Sales"])
print(f"Filas originales: {df_sales.count()}")
print(f"Filas despues de dropna: {df_sin_nulls.count()}")
print(f"Filas eliminadas: {df_sales.count() - df_sin_nulls.count()}")

In [ ]:
# Estrategia 2: Rellenar nulls con valores por defecto
df_rellenado = df_sales.fillna({
    "Weekly_Sales": 0,
    "IsHoliday": False
})

# Verificar que ya no hay nulls en esas columnas
print("Nulls despues de fillna:")
df_rellenado.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_rellenado.columns]
).show()

## 4. Detectar y Eliminar Duplicados

Los duplicados pueden distorsionar completamente los resultados de un analisis.

In [ ]:
# Detectar duplicados
total = df_sales.count()
sin_duplicados = df_sales.dropDuplicates().count()

print(f"Total de filas: {total}")
print(f"Filas unicas: {sin_duplicados}")
print(f"Duplicados encontrados: {total - sin_duplicados}")

In [ ]:
# Tambien podemos buscar duplicados en columnas especificas
# Por ejemplo, duplicados por Store + Dept + Date
sin_dup_parcial = df_sales.dropDuplicates(["Store", "Dept", "Date"]).count()
print(f"Filas unicas por (Store, Dept, Date): {sin_dup_parcial}")
print(f"Duplicados parciales: {total - sin_dup_parcial}")

## 5. Conversion de Tipos (Casting)

A veces los tipos inferidos no son correctos. Podemos convertirlos manualmente.

In [ ]:
# Ejemplo: convertir columna Date de string a tipo fecha
df_con_fecha = df_sales.withColumn(
    "Date_parsed", 
    F.to_date("Date", "yyyy-MM-dd")
)

# Verificar el cambio
df_con_fecha.select("Date", "Date_parsed").show(5)
df_con_fecha.printSchema()

In [ ]:
# Otros castings comunes
df_cast = df_sales \
    .withColumn("Weekly_Sales_int", F.col("Weekly_Sales").cast(IntegerType())) \
    .withColumn("Store_str", F.col("Store").cast(StringType()))

df_cast.select("Weekly_Sales", "Weekly_Sales_int", "Store", "Store_str").show(5)
df_cast.printSchema()

## 6. Funciones Definidas por el Usuario (UDFs)

Las UDFs permiten aplicar logica personalizada de Python a cada fila del DataFrame.

**Importante:** Las UDFs son mas lentas que las funciones nativas de Spark porque:
1. Serializan datos de JVM a Python y viceversa
2. Spark no puede optimizar el codigo Python

Usar solo cuando no hay una funcion nativa equivalente.

In [ ]:
from pyspark.sql.functions import udf

# Definir una UDF que categoriza el monto de ventas
@udf(returnType=StringType())
def categorizar_venta(monto):
    if monto is None:
        return "Sin datos"
    elif monto > 50000:
        return "Alta"
    elif monto > 20000:
        return "Media"
    else:
        return "Baja"

# Aplicar la UDF
df_categorizado = df_sales.withColumn("categoria", categorizar_venta("Weekly_Sales"))
df_categorizado.select("Store", "Dept", "Weekly_Sales", "categoria").show(10)

In [ ]:
# Ver la distribucion de categorias
df_categorizado.groupBy("categoria").count().orderBy("count", ascending=False).show()

## 7. Alternativa Nativa: when/otherwise

Para logica condicional simple, **siempre preferir `F.when().otherwise()`** sobre UDFs. Es mas rapido porque Spark puede optimizarlo internamente.

In [ ]:
# Misma logica que la UDF pero con funciones nativas (mas rapido)
df_nativo = df_sales.withColumn(
    "categoria",
    F.when(F.col("Weekly_Sales").isNull(), "Sin datos")
     .when(F.col("Weekly_Sales") > 50000, "Alta")
     .when(F.col("Weekly_Sales") > 20000, "Media")
     .otherwise("Baja")
)

df_nativo.select("Store", "Dept", "Weekly_Sales", "categoria").show(10)

# Verificar que el resultado es el mismo
df_nativo.groupBy("categoria").count().orderBy("count", ascending=False).show()

## 8. Combinar DataFrames: Join y Union

### Join
Combina columnas de dos DataFrames basandose en una condicion (como SQL JOIN).

### Union
Apila las filas de dos DataFrames con el mismo schema (como SQL UNION ALL).

In [ ]:
# JOIN: combinar sales con stores para obtener informacion de la tienda
df_completo = df_sales.join(df_stores, on="Store", how="inner")

print("Schema despues del join:")
df_completo.printSchema()
df_completo.show(5)
print(f"Filas despues del join: {df_completo.count()}")

In [ ]:
# UNION: apilar DataFrames
# Ejemplo: dividir y volver a unir (para demostrar la sintaxis)
df_parte1 = df_sales.filter(F.col("Store") <= 22)
df_parte2 = df_sales.filter(F.col("Store") > 22)

print(f"Parte 1: {df_parte1.count()} filas")
print(f"Parte 2: {df_parte2.count()} filas")

df_unido = df_parte1.union(df_parte2)
print(f"Despues de union: {df_unido.count()} filas")
print(f"Original: {df_sales.count()} filas")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Limpiar flights.csv - eliminar filas con nulls
# =============================================================
# TODO: 
#   1. Lee flights.csv con inferSchema=True
#   2. Cuenta los nulls en DEPARTURE_DELAY y ARRIVAL_DELAY
#   3. Elimina las filas donde DEPARTURE_DELAY o ARRIVAL_DELAY sean null
#      (usa dropna con subset)
#   4. Muestra cuantas filas se eliminaron
#
# Pista: 
#   filas_antes = df.count()
#   df_limpio = df.dropna(subset=["DEPARTURE_DELAY", "ARRIVAL_DELAY"])
#   filas_despues = df_limpio.count()
#   print(f"Filas eliminadas: {filas_antes - filas_despues}")

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Normalizar columnas de stores.csv
# =============================================================
# TODO:
#   1. Lee stores.csv
#   2. Renombra las columnas a snake_case:
#      - Store -> store
#      - Type -> tipo
#      - Size -> tamano
#      (usa .withColumnRenamed("viejo", "nuevo"))
#   3. Convierte la columna "tipo" a minusculas
#      (usa F.lower())
#   4. Muestra el resultado con show()
#
# Pista:
#   df.withColumnRenamed("Store", "store")
#     .withColumn("tipo", F.lower(F.col("tipo")))

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: UDF para categorizar distancia de vuelos
# =============================================================
# TODO:
#   1. Lee flights.csv
#   2. Crea una UDF llamada categorizar_distancia que reciba
#      la distancia y retorne:
#      - "corto" si distancia < 500
#      - "medio" si distancia >= 500 y <= 1500
#      - "largo" si distancia > 1500
#      - "desconocido" si es None
#   3. Aplica la UDF creando una nueva columna "tipo_vuelo"
#   4. Muestra la distribucion con groupBy("tipo_vuelo").count()
#
# Pista:
#   @udf(returnType=StringType())
#   def categorizar_distancia(distancia):
#       ...

# Escribe tu codigo aqui:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Crea un pipeline completo de limpieza para `netflix_titles.csv`.

In [ ]:
# =============================================================
# DESAFIO: Pipeline completo de limpieza de netflix_titles.csv
# =============================================================
# TODO: Realiza las siguientes transformaciones en orden:
#   1. Lee netflix_titles.csv
#   2. Maneja nulls:
#      - director: rellenar con "Desconocido"
#      - cast: rellenar con "No especificado"
#      - country: rellenar con "No especificado"
#   3. Parsea date_added a tipo fecha (formato "Month dd, yyyy")
#      Pista: F.to_date("date_added", "MMMM d, yyyy")
#   4. Separa "duration" en dos columnas:
#      - duration_valor (numerico): extraer el numero
#      - duration_unidad (string): extraer "min" o "Season"
#      Pista: F.split("duration", " ")
#   5. Elimina duplicados por show_id
#   6. Muestra el schema final y las primeras 10 filas
#   7. Muestra un conteo de nulls para verificar la limpieza

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Detectar nulls:** `F.count(F.when(F.isnull(c), c))` para contar nulls por columna
2. **Manejar nulls:** `dropna(subset=[...])` para eliminar, `fillna({...})` para rellenar
3. **Duplicados:** `dropDuplicates()` para eliminar registros repetidos
4. **Casting:** `F.to_date()`, `.cast()` para convertir tipos de datos
5. **UDFs:** `@udf(returnType=...)` para logica personalizada (usar con moderacion)
6. **when/otherwise:** Alternativa nativa y mas rapida que UDFs para logica condicional
7. **Join y Union:** Combinar DataFrames por columnas (join) o apilar filas (union)

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")